In [132]:
import pandas as pd

df = pd.read_excel('./data/Skylines_GameData.xlsx', sheet_name="Skylines_GameData")


In [133]:
df.head()

,anon_id,AGE,GENDER,LOCATION,EXPERTISE,GAME_REVISION,CITY,SESSION_ID,game,YEAR,...,OPTION_3,OPTION_4,env_before_policy,health_before_policy,satisf_before_policy,ecny_before_policy,env_after_policy,health_after_policy,satisf_after_policy,ecny_after_policy
0,u_1,50-64,Male,Amsterdam,4,4003,Amsterdam,wPXclC4pWk,6445.0,1,...,NaN,NaN,12.0,36.0,26.0,19.0,14.0,34.0,27.0,19.0
1,u_1,50-64,Male,Amsterdam,4,4003,Amsterdam,wPXclC4pWk,6445.0,2,...,NaN,NaN,14.0,34.0,27.0,19.0,10.0,30.0,28.0,18.0
2,u_1,50-64,Male,Amsterdam,4,4003,Amsterdam,wPXclC4pWk,6445.0,3,...,NaN,NaN,10.0,30.0,28.0,18.0,14.0,32.0,29.0,18.0
3,u_1,50-64,Male,Amsterdam,4,4003,Amsterdam,wPXclC4pWk,6445.0,4,...,NaN,NaN,14.0,32.0,29.0,18.0,22.0,39.0,22.0,12.0
4,u_1,50-64,Male,Amsterdam,4,4003,Amsterdam,wPXclC4pWk,6445.0,5,...,Encourage use of heat pumps to lower fuel bills,Subsidise people to replace old cars with hybrids,22.0,39.0,22.0,12.0,22.0,39.0,22.0,12.0


In [134]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173516 entries, 0 to 173515
Data columns (total 30 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   anon_id               173516 non-null  object 
 1   AGE                   173516 non-null  object 
 2   GENDER                173516 non-null  object 
 3   LOCATION              173516 non-null  object 
 4   EXPERTISE             173516 non-null  object 
 5   GAME_REVISION         173516 non-null  int64  
 6   CITY                  173516 non-null  object 
 7   SESSION_ID            173125 non-null  object 
 8   game                  135920 non-null  float64
 9   YEAR                  173516 non-null  int64  
 10  AREA                  173516 non-null  object 
 11  TYPE                  173516 non-null  object 
 12  CHOSEN_ID             173516 non-null  int64  
 13  OPTION_1_ID           173516 non-null  int64  
 14  OPTION_2_ID           173516 non-null  int64  
 15  

Lots of missing values, that we will treat right now. 
We won't drop any columns, as all of them can be source of further discussion.

In [135]:
BEFORE_POLICY = "_before_policy"
AFTER_POLICY = "_after_policy"

In [136]:
def calc_completed(serie, suffix) :
    return sum([1 for elem in [serie["env"+suffix], 
            serie["health"+suffix], 
            serie["satisf"+suffix], 
            serie["ecny"+suffix]] if not pd.isna(elem)])    
# spent way too much time realizing that imported data is stored as np.float first
# and that checking "is None" never hit

In [137]:
completeness_rows = []

for anon_id, group in df.groupby("anon_id", sort=False):
    before_completed = sum(
        calc_completed(row, BEFORE_POLICY)
        for _, row in group.iterrows()
    )
    after_completed = sum(
        calc_completed(row, AFTER_POLICY)
        for _, row in group.iterrows()
    )

    before_total = len(group) * 4
    after_total = len(group) * 4

    completeness_rows.append({
        "anon_id": anon_id,
        "before_completed": before_completed,
        "before_total": before_total,
        "after_completed": after_completed,
        "after_total": after_total,
    })

completeness = pd.DataFrame(completeness_rows)

completeness["before_percent"] = completeness["before_completed"] / completeness["before_total"]
completeness["after_percent"] = completeness["after_completed"] / completeness["after_total"]
completeness["ba_percent"] = (completeness["before_completed"] + completeness["after_completed"]) / (completeness["before_total"] + completeness["after_total"]) 

In [138]:
total = 0
for index, row in completeness.iterrows() :
    if row["after_percent"] <= 0.8 : 
        total += 1
total

865

In [139]:
total = 0
for index, row in completeness.iterrows() : 
    if row["before_percent"] <= 0.8 :
        total += 1

total

865

We end up with around 865 users that are 80% full, so we might as well drop them. We can try to look up for a correlation between missing values and other variables but it would take a long time.

In [140]:
SCORE_COLS = [c for c in df.columns if c.endswith(("_before_policy", "_after_policy"))]
m = df[SCORE_COLS].isna()

In [141]:
m.sum()

env_before_policy       37596
health_before_policy    37596
satisf_before_policy    37596
ecny_before_policy      37596
env_after_policy        37596
health_after_policy     37596
satisf_after_policy     37596
ecny_after_policy       37596
dtype: int64

I might just drop all of them, as they are not of any use for us, as missing values can be tough to reproduce. We would need to do some machine learning to understand the variations of each options, and then replicate them to fill the gaps, which, as you can notice, is quite a pain to deal with.

In [142]:
df_clean = df.dropna(axis=0, subset=SCORE_COLS, how="any")
# "how" means if there is a NA, drop the line/col
# axis = 0 -> line / axis = 1 -> col

In [143]:
m = df_clean[SCORE_COLS].isna()
m.sum()

env_before_policy       0
health_before_policy    0
satisf_before_policy    0
ecny_before_policy      0
env_after_policy        0
health_after_policy     0
satisf_after_policy     0
ecny_after_policy       0
dtype: int64

We come to the end of this cleaning. It was a pain to deal with this data, but it is now safe to say that we should drop it.

Duplicates are crucial and should be kept, as they highlight similar profiles between players, which is part of our objective.

## Mapping Variables

We have tons of variables that are simply plain text, ranging from sessions ids, to desription of options. Whilst description of options are long, we can create a dictionnary to map the option number to its description, save it in a file and re access it later.

Naturally, the same logic applies to the variables like Location, City, Gender, Age, Area and Type, where the number is generated. The best format to register this information should be a JSON.

Let's start with this.

In [144]:
def map_variable(name : str, col : pd.Series) :
    """Creates a map with each key being the number the value was mapped to.
    The structure looks like this : 
    
    name : {
        "0" : val1,
        "1" : val2,
        ...
    }

    Args:
        name (str): _description_
        col (pd.Series): _description_

    Returns:
        _type_: _description_
    """
    map_dict = {name : dict()}
    
    i = 0
    uniques = col.unique()
    for item in uniques: 
        map_dict[name][i] = item
        i += 1

    return map_dict

def map_options(name : str, df : pd.DataFrame, cols : list) :
    """Creates a single map shared by every option column, with each key being the
    ID the game already gave to the option, and the value being its description.
    CHOSEN_ID and OPTION_x_ID all use the same IDs, so one map is enough.
    The structure looks like this : 

    name : {
        "id1" : description1,
        "id2" : description2,
        ...
    }

    Args:
        name (str): name of the map in the json
        df (pd.DataFrame): dataframe holding both the text and the _ID columns
        cols (list): text columns, each one must have a matching col + "_ID" column

    Returns:
        dict: the map
    """
    map_dict = {name : dict()}

    for col in cols :
        col_id = col + "_ID"
        # OPTION_3 and OPTION_4 are not always proposed, hence the NaN
        df_filtered : pd.DataFrame = df[[col_id, col]].dropna().drop_duplicates()

        for index, row in df_filtered.iterrows() :
            # OPTION_3_ID and OPTION_4_ID are floats because of the NaN
            option_id = int(row[col_id])
            if option_id in map_dict[name] and map_dict[name][option_id] != row[col] :
                print(f"ID {option_id} has more than one description ! Keeping the first one")
                continue
            map_dict[name][option_id] = row[col]

    return map_dict

def fuse_large_json(maps : list) :
    fuse_dict = dict()
    for map in maps :
        name = list(map.keys())[0]
        fuse_dict[name] = map[name]
    return fuse_dict

In [145]:
import json
def create_json():
    try :
        with open("variable_map.json", 'x') as file : 
            file.write("")
    except FileExistsError :
        print("File already exists ! Check the repo root under the name 'variable_map.json'")

def map_to_json(map_dict : dict):
    try :
        with open("variable_map.json", 'w') as file :
            json.dump(map_dict, file)
    except FileNotFoundError :
        print("File not found ! Couldn't write to it. Try calling 'create_json before using this function.'")
    except Exception as e :
        print("Something went terribly wrong, take a look at the function")
        print(e)

In [146]:
create_json()

File already exists ! Check the repo root under the name 'variable_map.json'


In [147]:
COL_MAP = ["LOCATION", 
           "CITY", 
           "GENDER", 
           "AGE", 
           "AREA", 
           "TYPE"]
OPTION_COLS = ["CHOSEN",
               "OPTION_1",
               "OPTION_2",
               "OPTION_3",
               "OPTION_4"]

# the _ID columns are already numbers, so we only need the ID -> description map
maps = [map_options("OPTIONS", df_clean, OPTION_COLS)]
df_mapped = df_clean.copy()

for col in COL_MAP :
    curr_map = map_variable(col, df_clean[col])

    # i need the other way around, keys being the full name of cities or areas
    inv_map = {v: k for k, v in curr_map[col].items()}
    # used to do an "=" but was replacing every previous column
    # now it specifies the row
    df_mapped[col] = df_mapped[col].map(inv_map)

    maps.append(curr_map)

# descriptions can be found back in the json with the _ID columns
df_mapped = df_mapped.drop(columns=OPTION_COLS)

fuse_dict = fuse_large_json(maps)

map_to_json(fuse_dict)

In [ ]:
df_mapped[COL_MAP]

,LOCATION,CITY,GENDER,AGE,AREA,TYPE
0,0,0,0,0,0,0
1,0,0,0,0,0,0
2,0,0,0,0,0,0
3,0,0,0,0,0,0
4,0,0,0,0,1,1
...,...,...,...,...,...,...
173511,4,4,0,4,4,0
173512,4,4,0,4,2,0
173513,4,4,0,4,1,1
173514,4,4,0,4,1,1


In [152]:
df.describe()

,GAME_REVISION,game,YEAR,CHOSEN_ID,OPTION_1_ID,OPTION_2_ID,OPTION_3_ID,OPTION_4_ID,env_before_policy,health_before_policy,satisf_before_policy,ecny_before_policy,env_after_policy,health_after_policy,satisf_after_policy,ecny_after_policy
count,173516.000000,135920.000000,173516.000000,173516.000000,173516.000000,173516.000000,60400.000000,60400.000000,135920.000000,135920.000000,135920.000000,135920.000000,135920.000000,135920.000000,135920.000000,135920.000000
mean,3540.977668,4705.005327,9.822483,186.211087,250.918901,199.926670,190.940050,191.448013,30.185837,47.390862,34.273462,17.086102,30.184410,47.390185,34.272911,17.085992
std,730.905854,2689.278389,7.828846,131.004635,150.658281,134.496967,133.509246,134.233762,21.011612,18.307908,18.199391,11.752265,21.011149,18.307687,18.199035,11.752020
min,2200.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2800.000000,2388.000000,5.000000,81.000000,107.000000,87.000000,82.000000,83.000000,16.000000,36.000000,24.000000,9.000000,15.000000,36.000000,24.000000,9.000000
50%,4003.000000,4714.000000,8.000000,165.000000,263.000000,182.000000,173.000000,174.000000,24.000000,44.000000,30.000000,16.000000,24.000000,44.000000,30.000000,16.000000
75%,4003.000000,7030.250000,14.000000,262.000000,382.000000,300.000000,279.000000,280.000000,38.000000,55.000000,40.000000,21.000000,38.000000,55.000000,40.000000,21.000000
max,4003.000000,9375.000000,69.000000,508.000000,507.000000,508.000000,508.000000,508.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


Hurray, the values are correctly mapped, I can finally stop to torture myself. Thank Claude for finding out that the column was disappearing when I was using groupby cuz I totally forgot about this and it took me an hour to realize.